In [2]:
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from rich import print
from rich.logging import RichHandler
from rich.table import Table
from rich.console import Console
import time
import os
import logging
import warnings

warnings.filterwarnings("ignore")

logger = logging.getLogger()
logger.setLevel(logging.INFO)

console_handler = RichHandler()
console_handler.setLevel(logging.INFO)

file_handler = logging.FileHandler("app.log", mode="a", encoding="utf-8")
file_handler.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

console = Console()

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_API_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT")
AZURE_OPENAI_API_DEPLOYMENT_4O = os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O")

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_API_ENDPOINT,
    deployment_name=AZURE_OPENAI_API_DEPLOYMENT_4O,
    openai_api_version=AZURE_OPENAI_API_VERSION,
    openai_api_key=AZURE_OPENAI_API_KEY
)



# PromptTemplate + LLM + OutputParser

In [7]:
# 1. prompt template
prompt = PromptTemplate.from_template("What is the capital of {country}?")

# 2. model

# 3. parser
parser = StrOutputParser()

# 4. chain
chain = prompt | llm | parser
# chain = prompt | llm

# 5. invoke
result = chain.invoke({"country":"France"})

print(result)

[04/09/26 16:13:50] INFO     HTTP Request: POST                                                     ]8;id=121012;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=789792;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

The capital of France is Paris.

# ChatPromptTemplate 

In [5]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "Translate {text} to Chinese")
])

chain = prompt | llm | parser

result = chain.invoke({"text": "I love programming."})

print(result)

[04/09/26 16:11:59] INFO     HTTP Request: POST                                                     ]8;id=467174;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=362079;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

"I love programming." can be translated to Chinese as:

我喜欢编程。 (Simplified Chinese)

Or in Traditional Chinese:

我喜歡編程。

# RunnableSequence (等價寫法) 跟|是一樣的，指示比較舊派寫法

In [9]:
from langchain_core.runnables import RunnableSequence

chain = RunnableSequence(prompt, llm, parser)

result = chain.invoke({"country": "Japan"})

print(result)

[04/09/26 16:14:01] INFO     HTTP Request: POST                                                     ]8;id=68634;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=110617;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

The capital of Japan is **Tokyo**.

# 多輸入變數

In [10]:
prompt = PromptTemplate.from_template(
    "Translate {text} to {language}"
)

chain = prompt | llm | parser

chain.invoke({
    "text": "Hello",
    "language": "Japanese"
})



[04/09/26 16:15:28] INFO     HTTP Request: POST                                                     ]8;id=899241;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=708066;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

'"Hello" in Japanese is **こんにちは** (pronounced *konnichiwa*).'

# Structured Output

In [11]:
from langchain_core.output_parsers import JsonOutputParser
parser = JsonOutputParser()

prompt = PromptTemplate.from_template(
    "Give me a JSON with name and age of a person about {topic}"
)

chain = prompt | llm | parser

result = chain.invoke({"topic": "a programmer"})

print(result)

[04/09/26 16:17:28] INFO     HTTP Request: POST                                                     ]8;id=599985;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=955781;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

{'name': 'Alice Johnson', 'age': 27}

In [12]:
from langchain_core.output_parsers import JsonOutputParser
parser = JsonOutputParser()

prompt = PromptTemplate.from_template(
    "Give me a JSON with name and age of a person about {topic}"
)

chain = prompt | llm 

result = chain.invoke({"topic": "a programmer"})

print(result)

[04/09/26 16:17:41] INFO     HTTP Request: POST                                                     ]8;id=588365;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=5869;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

AIMessage(
    content='Sure! Here is a JSON object representing a programmer with their name and age:\n\n```json\n{\n  
"name": "Alex Johnson",\n  "age": 29\n}\n```',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 38,
            'prompt_tokens': 21,
            'total_tokens': 59,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-4.1-2025-04-14',
        'system_fingerprint': 'fp_7a7fd0eb44',
        'id': 'chatcmpl-DSes1Ex7SlUHtAA9yNUUypJkS7A7N',
        'prompt_filter_results': [
            {
                'prompt_index': 0,
                'content_filter_results': {
                    'hate': {'filtered': False, 'severity': 'safe'},
                    'jailbreak': {'detected': False, 'filtered': False},
                    'self_harm': {'filtered': False, 'severity': 'safe'},
                    'sexual': {'filtered': False, 'severity': 'safe'},
                    'violence': {'filtered': False, 'severity': 'safe'}
                }
            }
        ],
        'finish_reason': 'stop',
        'logprobs': None,
        'content_filter_results': {
            'hate': {'filtered': False, 'severity': 'safe'},
            'protected_material_code': {'detected': False, 'filtered': False},
            'protected_material_text': {'detected': False, 'filtered': False},
            'self_harm': {'filtered': False, 'severity': 'safe'},
            'sexual': {'filtered': False, 'severity': 'safe'},
            'violence': {'filtered': False, 'severity': 'safe'}
        }
    },
    id='lc_run--019d7151-81de-7571-87d3-11b11fcdcd60-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 21,
        'output_tokens': 38,
        'total_tokens': 59,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

# ChatPromptTemplate advanced usage


In [13]:
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Translate 'hello' to Chinese")
]

response = llm.invoke(messages)
print(response)



[04/09/26 16:24:27] INFO     HTTP Request: POST                                                     ]8;id=496307;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=919412;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

AIMessage(
    content='"Hello" in Chinese is 你好 (nǐ hǎo).',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 17,
            'prompt_tokens': 23,
            'total_tokens': 40,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-4.1-2025-04-14',
        'system_fingerprint': 'fp_7a7fd0eb44',
        'id': 'chatcmpl-DSeyYVrpDg8LjhMXnb3Iv4DVRxaxp',
        'prompt_filter_results': [
            {
                'prompt_index': 0,
                'content_filter_results': {
                    'hate': {'filtered': False, 'severity': 'safe'},
                    'jailbreak': {'detected': False, 'filtered': False},
                    'self_harm': {'filtered': False, 'severity': 'safe'},
                    'sexual': {'filtered': False, 'severity': 'safe'},
                    'violence': {'filtered': False, 'severity': 'safe'}
                }
            }
        ],
        'finish_reason': 'stop',
        'logprobs': None,
        'content_filter_results': {
            'hate': {'filtered': False, 'severity': 'safe'},
            'protected_material_code': {'detected': False, 'filtered': False},
            'protected_material_text': {'detected': False, 'filtered': False},
            'self_harm': {'filtered': False, 'severity': 'safe'},
            'sexual': {'filtered': False, 'severity': 'safe'},
            'violence': {'filtered': False, 'severity': 'safe'}
        }
    },
    id='lc_run--019d7157-b0cd-7f32-aae3-6e9f01370ffc-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 23,
        'output_tokens': 17,
        'total_tokens': 40,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

In [17]:
prompt = ChatPromptTemplate.from_messages([
    ("system","You are a {role}"),
    ("user", "Explain {topic} simply")
])

chain = prompt | llm 

result = chain.invoke({
    "role": "teacher",
    "topic": "neural networks"
})

print(result)

[04/09/26 16:26:32] INFO     HTTP Request: POST                                                     ]8;id=899498;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=902669;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

AIMessage(
    content='Sure! Here’s a simple explanation:\n\n**Neural networks** are a type of computer program inspired by 
the human brain.\n\nImagine a neural network as a group of connected dots (called "neurons"). Each dot receives 
some information, does some calculation, and then passes the result to the next dots.\n\nHow it works:\n1. 
**Input:** You give it some data (like a picture or some numbers).\n2. **Processing:** The network has layers of 
neurons. Each layer changes the data a little based on what it has learned.\n3. **Output:** It gives you an answer 
(like whether the picture is a cat or a dog).\n\nNeural networks learn by looking at lots of examples. At first, 
their answers may be wrong, but each time, they adjust how the dots are connected (called "weights") to 
improve.\n\nIn summary:\n- **Neural networks** are like connected calculators.\n- They can learn patterns in data 
by adjusting themselves.\n- They’re used for tasks like recognizing images, understanding speech, and making 
predictions.\n\nLet me know if you’d like more detail or an example!',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 226,
            'prompt_tokens': 19,
            'total_tokens': 245,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-4.1-2025-04-14',
        'system_fingerprint': 'fp_7a7fd0eb44',
        'id': 'chatcmpl-DSf0YLQ5k892TqlueLJ70EdfoYxmT',
        'prompt_filter_results': [
            {
                'prompt_index': 0,
                'content_filter_results': {
                    'hate': {'filtered': False, 'severity': 'safe'},
                    'jailbreak': {'detected': False, 'filtered': False},
                    'self_harm': {'filtered': False, 'severity': 'safe'},
                    'sexual': {'filtered': False, 'severity': 'safe'},
                    'violence': {'filtered': False, 'severity': 'safe'}
                }
            }
        ],
        'finish_reason': 'stop',
        'logprobs': None,
        'content_filter_results': {
            'hate': {'filtered': False, 'severity': 'safe'},
            'protected_material_code': {'detected': False, 'filtered': False},
            'protected_material_text': {'detected': False, 'filtered': False},
            'self_harm': {'filtered': False, 'severity': 'safe'},
            'sexual': {'filtered': False, 'severity': 'safe'},
            'violence': {'filtered': False, 'severity': 'safe'}
        }
    },
    id='lc_run--019d7159-9488-7171-87cc-fc5c0cc287cc-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 19,
        'output_tokens': 226,
        'total_tokens': 245,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

# MessagesPlaceholder

In [22]:
from langchain_core.prompts import MessagesPlaceholder

# output = StrOutputParser()

history = [
    HumanMessage(content="Hi"),
    AIMessage(content="Hello!")
]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{input}")
])

chain = prompt | llm 

result = chain.invoke({
    "history": history,
    "input": "What did I just say?"
})

print(result)

[04/09/26 16:32:58] INFO     HTTP Request: POST                                                     ]8;id=782139;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=828221;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

AIMessage(
    content='You just said "Hi."',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 7,
            'prompt_tokens': 34,
            'total_tokens': 41,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-4.1-2025-04-14',
        'system_fingerprint': 'fp_7a7fd0eb44',
        'id': 'chatcmpl-DSf6nq1wNSj4Jg9Abt4IwdI44oftr',
        'prompt_filter_results': [
            {
                'prompt_index': 0,
                'content_filter_results': {
                    'hate': {'filtered': False, 'severity': 'safe'},
                    'jailbreak': {'detected': False, 'filtered': False},
                    'self_harm': {'filtered': False, 'severity': 'safe'},
                    'sexual': {'filtered': False, 'severity': 'safe'},
                    'violence': {'filtered': False, 'severity': 'safe'}
                }
            }
        ],
        'finish_reason': 'stop',
        'logprobs': None,
        'content_filter_results': {
            'hate': {'filtered': False, 'severity': 'safe'},
            'protected_material_code': {'detected': False, 'filtered': False},
            'protected_material_text': {'detected': False, 'filtered': False},
            'self_harm': {'filtered': False, 'severity': 'safe'},
            'sexual': {'filtered': False, 'severity': 'safe'},
            'violence': {'filtered': False, 'severity': 'safe'}
        }
    },
    id='lc_run--019d715f-7ea5-7681-ad94-876144938039-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 34,
        'output_tokens': 7,
        'total_tokens': 41,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

In [24]:
summary_prompt = PromptTemplate.from_template("Summarize: {text}")

translate_prompt = PromptTemplate.from_template("Translate to Chinese: {text}")

chain = summary_prompt | llm | StrOutputParser() | translate_prompt | llm | StrOutputParser()

result = chain.invoke({"text": "LangChain is a powerful framework for building applications with language models. It provides tools for prompt management, chaining, and output parsing, making it easier to create complex interactions with LLMs."})

print(result)

[04/09/26 16:54:31] INFO     HTTP Request: POST                                                     ]8;id=53642;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=523043;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/09/26 16:54:32] INFO     HTTP Request: POST                                                     ]8;id=367027;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=635719;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

LangChain 
是一个简化使用语言模型构建应用程序的框架。它提供了管理提示、串联多个大语言模型操作以及解析输出的工具，让开发者能够
轻松创建高级且互动的解决方案。

# CommaSeparatedListOutputParser

In [26]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

prompt = PromptTemplate.from_template(
    "List 5 programming languages popular in 2025, separated by commas."
)

parser = CommaSeparatedListOutputParser()

chain = prompt | llm | parser

result = chain.invoke({})

print(result)

[04/09/26 18:31:48] INFO     HTTP Request: POST                                                     ]8;id=603874;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=34861;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

['Python', 'JavaScript', 'TypeScript', 'Java', 'Go']

# PydanticOutputParser

In [43]:
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

# 1️⃣ Define a Pydantic model
class Person(BaseModel):
    name: str
    age: int

# 2️⃣ Create the parser
parser = PydanticOutputParser(pydantic_object=Person)

# 3️⃣ Use the parser to get the format instructions
format_instructions = parser.get_format_instructions()
print("Format Instructions:", format_instructions)

# 4️⃣ Create the prompt with the instructions
prompt = PromptTemplate.from_template(
    "Give me the name and age of a famous AI researcher.\n"
    "Return the result strictly in JSON format:\n"
    # f"{format_instructions.replace('{', '{{').replace('}', '}}')}"  # escape braces
)

# 5️⃣ Build the chain
chain = prompt | llm | parser

# chain = prompt | llm

# 6️⃣ Invoke the chain
result = chain.invoke({})  # no need to pass anything else

# 7️⃣ Print the structured output
print(result)
# print(result.model_dump_json(indent=4))

Format Instructions: The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": 
"array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": 
["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"title": "Name", "type": "string"}, "age": {"title": "Age", "type": "integer"}}, 
"required": ["name", "age"]}
```

[04/09/26 18:48:04] INFO     HTTP Request: POST                                                     ]8;id=633085;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=921640;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

Person(name='Yoshua Bengio', age=59)

# RunnablePassthrough 

In [ ]:
# RunnablePassthrough => returns the input as output without any modification
from langchain_core.runnables import RunnablePassthrough

runnable = RunnablePassthrough()

result = runnable.invoke("Hello, World!")

print(result)

Hello, World!

# RunnableLambda

In [ ]:
# RunnableLambda => allows you to create a runnable from any lambda function
from langchain_core.runnables import RunnableLambda

upper_runnable = RunnableLambda(lambda x: x.upper())

print(upper_runnable.invoke("hello, world"))



HELLO, WORLD

# RunnableSequence

In [48]:
# RunnableSequence => allows you to chain multiple runnables together in a sequence
from langchain_core.runnables import RunnableSequence

seq = RunnableSequence(
    RunnableLambda(lambda x: x.strip()),
    RunnableLambda(lambda x: x.upper()),
    RunnableLambda(lambda x: f"Result: {x}")
)

print(seq.invoke("  hello, world  "))

Result: HELLO, WORLD